# **Indexing: Inverted Index**


In [1]:
!pip install python-terrier

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.7 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=6ead10edd6db4b84b896920eaae9d82d0bdc1a374e1d433366d80852a23165be
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
 

In [2]:
import pandas as pd
import pyterrier as pt
import os
import re

In [3]:
if not pt.java.started():
    pt.java.init()

terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [4]:
df=pd.read_csv('/content/movies_preprocessed.csv')
df.head()

,doc_id,title,type,year,rating,original_abstract,processed_text
0,doc_1,7 Dogs,movie,2026.0,0.000,Interpol agent Khalid Al-Azzazi joins forces w...,7 dog interpol agent khalid al azzazi join for...
1,doc_2,Karmouz War,movie,2018.0,5.942,"Alexandria, Egypt, 1940. Three young Egyptians...",karmouz war alexandria egypt 1940 three young ...
2,doc_3,EgyBest,movie,2026.0,0.000,Two friends dive into the digital underworld c...,egybest two friend dive digit underworld chase...
3,doc_4,Love Story,movie,2019.0,7.500,"Youssef, whose brother is trying to push him t...",love stori youssef whose brother tri push marr...
4,doc_5,Incident of Dishonor,movie,1971.0,0.000,One of the beauties of the estate lives with h...,incid dishonor one beauti estat live sister br...


In [5]:
df['docno']=df['doc_id'].astype(str)
df[['docno', 'processed_text']].head()

,docno,processed_text
0,doc_1,7 dog interpol agent khalid al azzazi join for...
1,doc_2,karmouz war alexandria egypt 1940 three young ...
2,doc_3,egybest two friend dive digit underworld chase...
3,doc_4,love stori youssef whose brother tri push marr...
4,doc_5,incid dishonor one beauti estat live sister br...


In [6]:
index_path=os.path.abspath("MoviesIndex")
os.makedirs(index_path, exist_ok=True)
indexer=pt.DFIndexer(index_path, overwrite=True)
index_ref=indexer.index(df["processed_text"], df["docno"])
index=pt.IndexFactory.of(index_ref)

/tmp/ipykernel_7872/3800516030.py:3: DeprecationWarning: Call to deprecated class DFIndexer. (use pt.terrier.IterDictIndexer().index(dataframe.to_dict(orient='records')) instead) -- Deprecated since version 0.11.0.
  indexer=pt.DFIndexer(index_path, overwrite=True)


In [7]:
print(index_ref.toString())

/content/MoviesIndex/data.properties


In [8]:
print(index.getCollectionStatistics().toString())

Number of documents: 10042
Number of terms: 21408
Number of postings: 258373
Number of fields: 0
Number of tokens: 289057
Field names: []
Positions:   false



In [9]:
inverted_index={}
for term_entry in index.getLexicon():
    term=term_entry.getKey()
    if term.isdigit() or re.match(r'^\d+\w*$', term):
        continue
    pointer=term_entry.getValue()
    doc_list=[]
    for posting in index.getInvertedIndex().getPostings(pointer):
        doc_id=posting.getId()
        frequency=posting.getFrequency()
        doc_list.append({'doc_id': doc_id, 'frequency': frequency})
    inverted_index[term]=doc_list

In [10]:
sample_terms=list(inverted_index.keys())[:10]
for term in sample_terms:
    print(f"Term: '{term}' -> {inverted_index[term]}")

Term: 'aa' -> [{'doc_id': 514, 'frequency': 1}, {'doc_id': 566, 'frequency': 1}, {'doc_id': 1318, 'frequency': 1}, {'doc_id': 4656, 'frequency': 1}, {'doc_id': 6243, 'frequency': 1}]
Term: 'aaa' -> [{'doc_id': 8076, 'frequency': 1}]
Term: 'aaan' -> [{'doc_id': 570, 'frequency': 1}]
Term: 'aad' -> [{'doc_id': 936, 'frequency': 1}, {'doc_id': 1724, 'frequency': 1}, {'doc_id': 2027, 'frequency': 1}, {'doc_id': 2675, 'frequency': 1}, {'doc_id': 3065, 'frequency': 1}]
Term: 'aaela' -> [{'doc_id': 812, 'frequency': 1}]
Term: 'aal' -> [{'doc_id': 2272, 'frequency': 2}, {'doc_id': 2296, 'frequency': 1}, {'doc_id': 3220, 'frequency': 1}, {'doc_id': 4768, 'frequency': 1}]
Term: 'aali' -> [{'doc_id': 3745, 'frequency': 1}]
Term: 'aallayl' -> [{'doc_id': 1571, 'frequency': 2}]
Term: 'aam' -> [{'doc_id': 5820, 'frequency': 1}]
Term: 'aan' -> [{'doc_id': 3065, 'frequency': 2}]


In [11]:
search_term="egypt"
if search_term in inverted_index:
    print(f"Documents containing '{search_term}':")
    print(inverted_index[search_term])
else:
    print(f"Term '{search_term}' not found in index")

Documents containing 'egypt':
[{'doc_id': 1, 'frequency': 1}, {'doc_id': 5, 'frequency': 1}, {'doc_id': 16, 'frequency': 1}, {'doc_id': 34, 'frequency': 1}, {'doc_id': 59, 'frequency': 2}, {'doc_id': 61, 'frequency': 1}, {'doc_id': 64, 'frequency': 1}, {'doc_id': 65, 'frequency': 1}, {'doc_id': 66, 'frequency': 1}, {'doc_id': 72, 'frequency': 1}, {'doc_id': 82, 'frequency': 1}, {'doc_id': 83, 'frequency': 1}, {'doc_id': 88, 'frequency': 1}, {'doc_id': 118, 'frequency': 1}, {'doc_id': 136, 'frequency': 2}, {'doc_id': 194, 'frequency': 1}, {'doc_id': 201, 'frequency': 1}, {'doc_id': 208, 'frequency': 2}, {'doc_id': 216, 'frequency': 1}, {'doc_id': 217, 'frequency': 1}, {'doc_id': 224, 'frequency': 1}, {'doc_id': 234, 'frequency': 1}, {'doc_id': 235, 'frequency': 1}, {'doc_id': 244, 'frequency': 1}, {'doc_id': 251, 'frequency': 1}, {'doc_id': 267, 'frequency': 1}, {'doc_id': 275, 'frequency': 1}, {'doc_id': 291, 'frequency': 1}, {'doc_id': 309, 'frequency': 1}, {'doc_id': 348, 'frequency'

In [12]:
if search_term in inverted_index:
    results=inverted_index[search_term]
    print(f"Term:'{search_term}' appears in {len(results)} documents\n")
    print("First 5 entries:")
    for entry in results[:5]:
        print(f"  doc_id: {entry['doc_id']}, frequency: {entry['frequency']}")

Term:'egypt' appears in 373 documents

First 5 entries:
  doc_id: 1, frequency: 1
  doc_id: 5, frequency: 1
  doc_id: 16, frequency: 1
  doc_id: 34, frequency: 1
  doc_id: 59, frequency: 2


In [13]:
total_terms=len(inverted_index)
total_documents=len(df)
print(f"Total unique terms: {total_terms}")
print(f"Total documents: {total_documents}")

Total unique terms: 21019
Total documents: 10042


In [14]:
import re
import pandas as pd
terms_stats=[]
for x in index.getLexicon():
    term=x.getKey()
    if term.isdigit() or re.match(r'^\d+\w*$', term):
        continue
    stats=x.getValue()
    terms_stats.append({
        'term':term,
        'frequency':stats.getFrequency(),
        'doc_count':stats.getDocumentFrequency()
    })
terms_df=pd.DataFrame(terms_stats)
terms_df.head(20)

,term,frequency,doc_count
0,aa,5,5
1,aaa,1,1
2,aaan,1,1
3,aad,5,5
4,aaela,1,1
5,aal,5,4
6,aali,1,1
7,aallayl,2,1
8,aam,1,1
9,aan,2,1


In [15]:
print("Sample Terms with Statistics:")
print(f"\n{'Term':<20} {'Total Frequency':<18} {'Document Count':<15}")
print("-" * 53)
for i in range(min(15, len(terms_df))):
    row = terms_df.iloc[i]
    print(f"{row['term']:<20} {row['frequency']:<18} {row['doc_count']:<15}")

Sample Terms with Statistics:

Term                 Total Frequency    Document Count 
-----------------------------------------------------
aa                   5                  5              
aaa                  1                  1              
aaan                 1                  1              
aad                  5                  5              
aaela                1                  1              
aal                  5                  4              
aali                 1                  1              
aallayl              2                  1              
aam                  1                  1              
aan                  2                  1              
aang                 1                  1              
aard                 1                  1              
aaron                10                 9              
aasar                1                  1              
aasr                 1                  1              


In [17]:
print("\nOriginal vs Processed Text Example:")
print(f"\nOriginal: {df.iloc[0]['original_abstract'][:100]}...")
print(f"\nProcessed: {df.iloc[0]['processed_text'][:100]}...")


Original vs Processed Text Example:

Original: Interpol agent Khalid Al-Azzazi joins forces with Ghali Abu Dawood from the 7 Dogs crime syndicate t...

Processed: 7 dog interpol agent khalid al azzazi join forc ghali abu dawood 7 dog crime syndic fight drug traff...
